# Air Quality Prediction and Pollution Episode Detection: ML model

A virtual-sensor model that **estimates the reference CO concentration (mg/m³) at hour t** from five low-cost
metal-oxide sensors, weather and earlier CO readings, evaluated on future periods. It then **flags and explains
pollution episodes** where the observed CO differs substantially from the estimate.

**Run it on Kaggle**
1. *Add Input* → search for `AirQualityUCI` and add any copy of the UCI Air Quality CSV (for example `chandanabonbon/airqualityuci-csv`).
   With *Internet* switched on, the notebook can also download the data from UCI by itself.
2. Optional: *Settings → Accelerator → GPU*. XGBoost then trains on CUDA; without a GPU it falls back to the CPU automatically.
3. *Run All*. Outputs (`air_quality_model.pkl`, `aq_features.py`, metrics and episodes) are written to the working folder.

The notebook follows the ML flow: problem definition → data collection → preprocessing → EDA → feature engineering →
model selection → training → evaluation → hyperparameter tuning → model saving.

## Shared feature code
The saved pipeline calls these functions, so they are written to `aq_features.py` and imported. The same file is used
by the FastAPI backend (`main.py`), which keeps training and serving identical.

In [ ]:
%%writefile aq_features.py
"""Shared feature engineering for training (train_model.py) and serving (main.py).

The saved pipeline references these functions, so this module must sit next to
main.py when the model is loaded.
"""
import numpy as np
import pandas as pd

# Raw inputs the API accepts (POST /predict) and the pipeline is trained on.
INPUT_COLUMNS = [
    "hour",               # 0-23
    "weekday",            # 0 = Monday ... 6 = Sunday
    "pt08_s1_co",         # low-cost tin-oxide sensor (CO targeted)
    "pt08_s2_nmhc",       # titania sensor (NMHC targeted)
    "pt08_s3_nox",        # tungsten-oxide sensor (NOx targeted)
    "pt08_s4_no2",        # tungsten-oxide sensor (NO2 targeted)
    "pt08_s5_o3",         # indium-oxide sensor (O3 targeted)
    "temperature",        # degrees C
    "relative_humidity",  # %
    "absolute_humidity",
    "co_lag_1",           # reference CO 1 hour earlier (mg/m3)
    "co_lag_2",
    "co_lag_3",
    "co_lag_24",          # same hour yesterday
    "pt08_s1_co_prev",    # each sensor's reading 1 hour earlier (optional)
    "pt08_s2_nmhc_prev",
    "pt08_s3_nox_prev",
    "pt08_s4_no2_prev",
    "pt08_s5_o3_prev",
]

SENSOR_INPUTS = ["pt08_s1_co", "pt08_s2_nmhc", "pt08_s3_nox", "pt08_s4_no2", "pt08_s5_o3"]

# Mapping from the UCI column names to the API field names.
SOURCE_TO_INPUT = {
    "PT08.S1(CO)": "pt08_s1_co",
    "PT08.S2(NMHC)": "pt08_s2_nmhc",
    "PT08.S3(NOx)": "pt08_s3_nox",
    "PT08.S4(NO2)": "pt08_s4_no2",
    "PT08.S5(O3)": "pt08_s5_o3",
    "T": "temperature",
    "RH": "relative_humidity",
    "AH": "absolute_humidity",
}

ENGINEERED_COLUMNS = [
    "hour_sin", "hour_cos", "weekday_sin", "weekday_cos",
    "pt08_s1_co", "pt08_s2_nmhc", "pt08_s3_nox", "pt08_s4_no2", "pt08_s5_o3",
    "temperature", "relative_humidity", "absolute_humidity",
    "co_lag_1", "co_lag_2", "co_lag_3", "co_lag_24",
    "co_lag_mean_3", "co_diff_1_2",
    "pt08_s1_co_prev", "pt08_s2_nmhc_prev", "pt08_s3_nox_prev", "pt08_s4_no2_prev", "pt08_s5_o3_prev",
    "pt08_s1_co_delta", "pt08_s2_nmhc_delta", "pt08_s3_nox_delta", "pt08_s4_no2_delta", "pt08_s5_o3_delta",
]


def add_engineered_features(X):
    """Turn raw API inputs into model features (cyclical time, lag summaries)."""
    X = pd.DataFrame(X, columns=INPUT_COLUMNS).astype(float)
    out = pd.DataFrame(index=X.index)
    out["hour_sin"] = np.sin(2 * np.pi * X["hour"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * X["hour"] / 24)
    out["weekday_sin"] = np.sin(2 * np.pi * X["weekday"] / 7)
    out["weekday_cos"] = np.cos(2 * np.pi * X["weekday"] / 7)
    for col in INPUT_COLUMNS[2:]:
        out[col] = X[col]
    for col in SENSOR_INPUTS:
        # Hour-to-hour sensor change tracks the change in CO (NaN if previous hour unknown).
        out[col + "_delta"] = X[col] - X[col + "_prev"]
    lags = X[["co_lag_1", "co_lag_2", "co_lag_3"]]
    # Mean of whichever recent lags exist; stays NaN (imputed later) if none exist.
    out["co_lag_mean_3"] = lags.mean(axis=1, skipna=True)
    out["co_diff_1_2"] = X["co_lag_1"] - X["co_lag_2"]
    return out[ENGINEERED_COLUMNS]


def engineered_feature_names(transformer, input_features=None):
    """feature_names_out hook for the FunctionTransformer (must be picklable)."""
    return np.array(ENGINEERED_COLUMNS, dtype=object)


def build_inputs(hourly, shift_sensors=0):
    """Build the INPUT_COLUMNS frame from the hourly UCI dataframe.

    shift_sensors=0 -> nowcast (sensors at hour t); 1 -> forecast (sensors at t-1).
    """
    co = hourly["CO(GT)"]
    X = pd.DataFrame(index=hourly.index)
    X["hour"] = hourly.index.hour
    X["weekday"] = hourly.index.day_of_week
    for src, dst in SOURCE_TO_INPUT.items():
        X[dst] = hourly[src].shift(shift_sensors)
    for lag in (1, 2, 3, 24):
        X[f"co_lag_{lag}"] = co.shift(lag)
    for src, dst in SOURCE_TO_INPUT.items():
        if dst in SENSOR_INPUTS:
            X[dst + "_prev"] = hourly[src].shift(shift_sensors + 1)
    return X[INPUT_COLUMNS]

## Setup

In [ ]:
import glob, io, json, os, time, warnings, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterGrid
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from xgboost import XGBRegressor
from aq_features import INPUT_COLUMNS, add_engineered_features, build_inputs, engineered_feature_names

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.figsize": (12, 4), "axes.grid": True, "grid.alpha": 0.3})
SEED, TARGET = 42, "CO(GT)"
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

## 1. Problem definition
* **Target:** reference CO concentration `CO(GT)` in mg/m³ at hour *t* (nowcast / virtual sensor).
* **Inputs:** sensor responses PT08.S1–S5 at *t* and *t−1*, temperature, humidity, hour and weekday, and CO at *t−1, t−2, t−3, t−24*.
* **Evaluation:** chronological splits only. Models are selected on validation folds; the test period is scored once.
* **Episodes:** hours where |observed − estimate| exceeds the 95th percentile of calibration residuals, classified as
  *unusual-pollution candidate* or *data-limitation candidate*.

## 2. Data collection

In [ ]:
REQUIRED = ["CO(GT)", "PT08.S1(CO)", "PT08.S2(NMHC)", "PT08.S3(NOx)", "PT08.S4(NO2)", "PT08.S5(O3)", "T", "RH", "AH"]

def read_table(source):
    """Read the original UCI format (';' and ',' decimals) or a comma-separated variant."""
    for kwargs in ({"sep": ";", "decimal": ","}, {}):
        if hasattr(source, "seek"):
            source.seek(0)
        table = pd.read_csv(source, **kwargs)
        table.columns = table.columns.str.strip()
        if "CO(GT)" in table.columns:
            return table
    raise ValueError("Could not find a CO(GT) column; is this the UCI Air Quality data?")

def load_raw():
    roots = ["/kaggle/input", "."]
    files = [f for r in roots for f in glob.glob(os.path.join(r, "**", "*"), recursive=True)
             if f.lower().endswith((".csv", ".zip")) and "air" in os.path.basename(f).lower()]
    files.sort(key=lambda f: ("airqualityuci" not in os.path.basename(f).lower().replace("_", ""), f.endswith(".zip")))
    for f in files:
        try:
            if f.lower().endswith(".zip"):
                z = zipfile.ZipFile(f)
                name = next(n for n in z.namelist() if n.lower().endswith(".csv"))
                table = read_table(io.BytesIO(z.read(name)))
            else:
                table = read_table(f)
            print("Using", f)
            return table
        except Exception as exc:
            print(f"Skipping {f}: {exc}")
    print("No local copy found; downloading from UCI ...")
    data = urllib.request.urlopen("https://archive.ics.uci.edu/static/public/360/air+quality.zip").read()
    return read_table(io.BytesIO(zipfile.ZipFile(io.BytesIO(data)).read("AirQualityUCI.csv")))

raw = load_raw()
print(raw.shape)
raw.head()

## 3. Data preprocessing
`-200` is the dataset's missing-value code. Rows are placed on a continuous hourly grid.

In [ ]:
df = raw.loc[:, ~raw.columns.str.startswith("Unnamed")].copy()
if {"Date", "Time"} <= set(df.columns):                      # original UCI layout
    df = df.dropna(subset=["Date", "Time"])
    ts = pd.to_datetime(df["Date"].astype(str) + " " + df["Time"].astype(str).str.replace(".", ":", regex=False),
                        dayfirst=True, errors="coerce")
    df = df.drop(columns=["Date", "Time"])
else:                                                         # pre-processed copy with one datetime column
    tcol = next((c for c in df.columns if c.lower() in ("datetime", "date_time", "timestamp", "date")), None)
    if tcol is None:
        raise ValueError(f"No Date/Time or datetime column found in {list(df.columns)}")
    ts = pd.to_datetime(df[tcol], errors="coerce")
    df = df.drop(columns=[tcol])
missing_cols = [c for c in REQUIRED if c not in df.columns]
assert not missing_cols, f"Dataset is missing required columns: {missing_cols}"
df = df.apply(pd.to_numeric, errors="coerce").replace(-200, np.nan).dropna(axis=1, how="all")
df.index = pd.DatetimeIndex(ts, name="timestamp")
df = df[df.index.notna()]
df = df[~df.index.duplicated()].sort_index().asfreq("h")
y = df[TARGET]
if y.notna().all():
    print("WARNING: CO(GT) has no missing values, so this copy was probably pre-filled (imputed).\n"
          "Metrics on imputed targets are not comparable with the original data. For honest results, "
          "use the original UCI file (data/air-quality.zip in the team bundle).")
n = len(df)
print(f"{n} hourly rows: {df.index.min()} -> {df.index.max()}")
missing = (df.isna().mean() * 100).round(2).sort_values(ascending=False)
missing.to_frame("missing %")

## 4. Exploratory data analysis

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
missing.plot.barh(ax=ax[0], title="Missing values (%)")
y.resample("D").mean().plot(ax=ax[1], title="Daily mean CO (mg/m³)")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
y.groupby(df.index.hour).mean().plot(ax=ax[0], marker="o", title="CO by hour of day")
y.groupby(df.index.day_of_week).mean().plot.bar(ax=ax[1], title="CO by weekday (0 = Mon)")
y.groupby(df.index.to_period("M")).mean().plot(ax=ax[2], marker="o", title="CO by month")
plt.tight_layout(); plt.show()

corr = df.corr()
plt.figure(figsize=(9, 7))
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.xticks(range(len(corr)), corr.columns, rotation=90); plt.yticks(range(len(corr)), corr.columns)
plt.colorbar(); plt.title("Correlation matrix"); plt.grid(False); plt.show()

## 5. Feature engineering
`build_inputs` creates CO lags, previous-hour sensor readings and the hour/weekday columns from the hourly data.
Inside the pipeline, `add_engineered_features` adds cyclical time, sensor deltas and lag summaries. Missing inputs are
imputed with training medians, plus missing-value indicators.

In [ ]:
X_now = build_inputs(df, shift_sensors=0)   # nowcast: sensors at hour t
X_fc = build_inputs(df, shift_sensors=1)    # forecast: sensors at t-1 (framing comparison)
sensor_ok = df["PT08.S1(CO)"].notna().to_numpy()
y_ok = y.notna().to_numpy()
base_ok = (X_now["co_lag_1"].notna() & X_now["co_lag_24"].notna()).to_numpy()

P = {k: int(n * v) for k, v in {"f1": .40, "f2": .50, "f3": .60, "cal": .70, "test": .80}.items()}
FOLDS = [("fold_1", P["f1"], P["f2"]), ("fold_2", P["f2"], P["f3"]), ("fold_3", P["f3"], P["cal"])]
START = 24

def train_idx(end):
    i = np.arange(START, end); return i[y_ok[i] & sensor_ok[i]]

def eval_idx(lo, hi):
    i = np.arange(lo, hi); return i[y_ok[i] & sensor_ok[i] & base_ok[i]]

def metrics(true, pred):
    true, pred = np.asarray(true, float), np.asarray(pred, float)
    return {"MAE": mean_absolute_error(true, pred), "RMSE": float(np.sqrt(mean_squared_error(true, pred))),
            "R2": r2_score(true, pred), "n": len(true)}

def mean_mae(rows):
    return float(np.mean([r["MAE"] for r in rows]))

def make_pipeline(estimator):
    return Pipeline([
        ("features", FunctionTransformer(add_engineered_features, feature_names_out=engineered_feature_names)),
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler()),
        ("model", estimator)])

for name, lo, hi in FOLDS + [("calibration", P["cal"], P["test"]), ("test", P["test"], n)]:
    print(f"{name:12s} {df.index[lo]} -> {df.index[hi-1]}")
X_now.head()

## 6. Model selection: candidates, defaults and tuning grids

In [ ]:
try:
    XGBRegressor(n_estimators=2, device="cuda").fit(np.random.rand(50, 3), np.random.rand(50))
    DEVICE = "cuda"
except Exception:
    DEVICE = "cpu"
print("XGBoost device:", DEVICE)

CANDIDATES = {
    "Linear Regression": (lambda **p: LinearRegression(), {}, None),
    "Ridge": (lambda **p: Ridge(**p), {"alpha": 1.0}, {"alpha": [0.1, 1, 3, 10, 30, 100]}),
    "KNN": (lambda **p: KNeighborsRegressor(**p), {"n_neighbors": 5, "weights": "uniform"},
            {"n_neighbors": [5, 10, 20, 40], "weights": ["uniform", "distance"]}),
    "Random Forest": (lambda **p: RandomForestRegressor(random_state=SEED, n_jobs=-1, **p), {"n_estimators": 100},
                      {"n_estimators": [200], "max_depth": [8, 14, None], "min_samples_leaf": [1, 5]}),
    "XGBoost": (lambda **p: XGBRegressor(random_state=SEED, device=DEVICE, tree_method="hist", verbosity=0, **p),
                {"n_estimators": 100},
                {"n_estimators": [300, 600], "max_depth": [3, 5], "learning_rate": [0.03, 0.08],
                 "subsample": [0.8], "colsample_bytree": [0.8]}),
}

def run_folds(name, params, X=X_now):
    rows = []
    for fold, lo, hi in FOLDS:
        tr, ev = train_idx(lo), eval_idx(lo, hi)
        m = make_pipeline(CANDIDATES[name][0](**params)).fit(X.iloc[tr], y.iloc[tr])
        rows.append({"fold": fold, **metrics(y.iloc[ev], m.predict(X.iloc[ev]))})
    return rows

## 7–9. Training, validation and hyperparameter tuning (validation folds only)

In [ ]:
results, tuning = {}, []
for name, (make, default, grid) in CANDIDATES.items():
    t0 = time.time()
    best_params, best_rows = default, run_folds(name, default)
    default_mae = mean_mae(best_rows)
    for params in (ParameterGrid(grid) if grid else []):
        rows = run_folds(name, params)
        if mean_mae(rows) < mean_mae(best_rows):
            best_params, best_rows = params, rows
    results[name] = {"params": best_params, "folds": best_rows}
    if grid:
        tuning.append({"model": name, "default MAE": default_mae, "tuned MAE": mean_mae(best_rows), "best params": best_params})
    print(f"{name:18s} validation MAE {mean_mae(best_rows):.4f}  {best_params}  ({time.time()-t0:.1f}s)")

SELECTED = min(results, key=lambda m: mean_mae(results[m]["folds"]))
print("\nSelected on validation only:", SELECTED, results[SELECTED]["params"])
pd.DataFrame(tuning)

## 8. Evaluation on the untouched test period (models frozen before the calibration period)

In [ ]:
tr_final = train_idx(P["cal"])
test_ev, cal_ev = eval_idx(P["test"], n), eval_idx(P["cal"], P["test"])
final_models = {name: make_pipeline(CANDIDATES[name][0](**r["params"])).fit(X_now.iloc[tr_final], y.iloc[tr_final])
                for name, r in results.items()}
final = final_models[SELECTED]

rows = []
for name, r in results.items():
    rows.append({"model": name, "type": "model", "validation MAE": mean_mae(r["folds"]),
                 **{f"test {k}": v for k, v in metrics(y.iloc[test_ev], final_models[name].predict(X_now.iloc[test_ev])).items()}})
for name, col in [("Persistence (previous hour)", "co_lag_1"), ("Seasonal naive (same hour yesterday)", "co_lag_24")]:
    val = np.mean([metrics(y.iloc[eval_idx(lo, hi)], X_now[col].iloc[eval_idx(lo, hi)])["MAE"] for _, lo, hi in FOLDS])
    rows.append({"model": name, "type": "baseline", "validation MAE": val,
                 **{f"test {k}": v for k, v in metrics(y.iloc[test_ev], X_now[col].iloc[test_ev]).items()}})
table = pd.DataFrame(rows).sort_values("validation MAE").reset_index(drop=True)
pers = table.loc[table.model.str.startswith("Persistence"), "test MAE"].iloc[0]
sel = table.loc[table.model == SELECTED, "test MAE"].iloc[0]
print(f"{SELECTED}: test MAE {sel:.3f} vs persistence {pers:.3f} -> {100*(1-sel/pers):.1f}% better")
table.round(4)

### Framing comparison: nowcast (sensors at t) vs forecast (sensors at t−1)

In [ ]:
framing = []
for label, X in [("nowcast", X_now), ("forecast", X_fc)]:
    rows_f = run_folds(SELECTED, results[SELECTED]["params"], X)
    m = make_pipeline(CANDIDATES[SELECTED][0](**results[SELECTED]["params"])).fit(X.iloc[tr_final], y.iloc[tr_final])
    framing.append({"framing": label, "validation MAE": mean_mae(rows_f),
                    "test MAE": metrics(y.iloc[test_ev], m.predict(X.iloc[test_ev]))["MAE"]})
pd.DataFrame(framing).round(4)

## Pollution episode detection and classification

In [ ]:
cal_res = np.abs(y.iloc[cal_ev].to_numpy() - final.predict(X_now.iloc[cal_ev]))
k = min(len(cal_res), int(np.ceil((len(cal_res) + 1) * 0.90)))
RADIUS = float(np.sort(cal_res)[k - 1])        # conformal 90% interval half-width
THRESHOLD = float(np.quantile(cal_res, 0.95))  # episode threshold
print(f"90% interval: estimate ± {RADIUS:.3f}   episode threshold: {THRESHOLD:.3f} mg/m³")

idx = np.arange(P["test"], n)[sensor_ok[P["test"]:]]
est = pd.Series(final.predict(X_now.iloc[idx]), index=df.index[idx])
obs = y.loc[est.index]
res = obs - est

pairs = df.iloc[START:P["cal"]][["PT08.S1(CO)", TARGET]].dropna()
coef = np.polyfit(pairs["PT08.S1(CO)"], pairs[TARGET], 1)
dis_thr = np.abs(pairs[TARGET] - np.polyval(coef, pairs["PT08.S1(CO)"])).quantile(0.99)
s_lo, s_hi = pairs["PT08.S1(CO)"].min(), pairs["PT08.S1(CO)"].max()
s1 = df.loc[est.index, "PT08.S1(CO)"]
imputed = X_now.loc[est.index, ["co_lag_1", "co_lag_2", "co_lag_3", "co_lag_24"]].isna().any(axis=1)
disagree = np.abs(obs - np.polyval(coef, s1)) > dis_thr
out_range = (s1 < s_lo) | (s1 > s_hi)
flag = (res.abs() > THRESHOLD) & obs.notna()

episodes, start, prev = [], None, None
for t in list(flag[flag].index) + [None]:
    if start is not None and (t is None or (t - prev) > pd.Timedelta(hours=3)):
        w = slice(start, prev)
        reasons = []
        if imputed[w].any(): reasons.append("missing CO history imputed")
        if out_range[w].any(): reasons.append("sensor outside training range")
        if disagree[w].any(): reasons.append("reference disagrees with sensor calibration line")
        episodes.append({"start": start, "end": prev, "hours": int(flag[w].sum()),
                         "peak |residual|": float(res[w].abs().max()),
                         "classification": "data-limitation candidate" if reasons else "unusual-pollution candidate",
                         "reason": "; ".join(reasons) or "inputs complete, sensors consistent"})
        start = None
    if t is not None:
        start = start or t
        prev = t
ep = pd.DataFrame(episodes)
print(ep["classification"].value_counts())
ep.head(15)

## Evaluation charts

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4.5))
ax.plot(obs.index, obs, lw=.7, label="observed")
ax.plot(est.index, est, lw=.7, label="estimated")
ax.fill_between(est.index, est - RADIUS, est + RADIUS, alpha=.2, label="90% interval")
ax.scatter(obs[flag].index, obs[flag], c="red", s=8, label="flagged", zorder=3)
ax.set_title(f"Test period: estimated vs observed CO ({SELECTED})"); ax.set_ylabel("mg/m³"); ax.legend()
plt.show()

fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
ok = obs.notna()
ax[0].scatter(obs[ok], est[ok], s=5, alpha=.4); lim = [0, max(obs.max(), est.max())]
ax[0].plot(lim, lim, "k--", lw=1); ax[0].set(title="Estimated vs observed", xlabel="observed", ylabel="estimated")
ax[1].hist(res.dropna(), bins=50); ax[1].set_title("Residuals (observed − estimated)")
imp = permutation_importance(final, X_now.iloc[cal_ev], y.iloc[cal_ev], scoring="neg_mean_absolute_error",
                             n_repeats=5, random_state=SEED)
pd.Series(imp.importances_mean, index=INPUT_COLUMNS).nlargest(12).sort_values().plot.barh(ax=ax[2])
ax[2].set_title("Permutation importance (MAE increase)")
plt.tight_layout(); plt.show()

## 10. Model saving
The model is saved for CPU serving, then reloaded and checked against the original.

In [ ]:
if SELECTED == "XGBoost":
    final.named_steps["model"].set_params(device="cpu")
path = os.path.join(OUT, "air_quality_model.pkl")
joblib.dump(final, path)
reloaded = joblib.load(path)
sample = X_now.iloc[test_ev[:20]]
assert np.allclose(reloaded.predict(sample), final.predict(sample), atol=1e-6)
table.to_csv(os.path.join(OUT, "model_comparison.csv"), index=False)
ep.to_csv(os.path.join(OUT, "test_episodes.csv"), index=False)
json.dump({"model": SELECTED, "params": results[SELECTED]["params"], "device_trained": DEVICE,
           "threshold": THRESHOLD, "interval_radius": RADIUS, "input_columns": INPUT_COLUMNS},
          open(os.path.join(OUT, "model_info.json"), "w"), indent=2)
print("Saved", path, "(+ model_comparison.csv, test_episodes.csv, model_info.json)")

## Try the saved model (same JSON fields as the API's `POST /predict`)

In [ ]:
model = joblib.load(path)
hour_t = obs.dropna().index[100]
example = X_now.loc[[hour_t]]
print("Input:", {k: (None if pd.isna(v) else v) for k, v in example.iloc[0].to_dict().items()})
print(f"Estimated CO: {model.predict(example)[0]:.3f} mg/m³   observed: {y[hour_t]:.3f} mg/m³")